%md
# Transform Circuits Data

1. Read bronze `circuits` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
1. Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)
1. Filter out rows where `circuit_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of columns `circuit_name` and `locality` to Title Case
1. Write the transformed data to silver `circuits` table

> Below changes are required to implement Incremental Load Processing
1. Accept batch_id as a parameter to the notebook
1. Process data for only the batch_id being passed in (i.e., filter reading from bronze using the batch_id)
1. Add created_timestamp, updated_timestamp and batch_id to the silver table. 
1. Merge the processed data to the silver table
    - created_timestamp should only be populated at the time of inserting/ creating the record. It should not be updated during the merge update.
    - Ensure that we are not overwriting the data in silver table by older bronze data (re-run scenario)

In [0]:
# 1. Get batch ID
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/03_silver_helpers"

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_path = "s3://bizf1project123/Bronze/Kalashri_increment/circuits"

silver_path = "s3://bizf1project123/Silver/Kalashri_increment/circuits"

## Step 1 - Read bronze races table

In [0]:
# 5. Read Bronze Delta data for the selected batch
circuits_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
    .filter(F.col("batch_id") == v_batch_id)
)

display(circuits_df)

In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

display(
    bronze_df.select("batch_id")
    .distinct()
)


## Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude"
        })
)

In [0]:
display(circuits_renamed_df)

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

display(circuits_valid_df)

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])
display(circuits_distinct_df)

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn('circuit_name', F.initcap(F.col("circuit_name")))
        .withColumn('locality', F.initcap(F.col("locality")))
)

display(circuits_final_df)

In [0]:

write_to_silver(
    input_df=circuits_final_df,
    target_path=silver_path,
    merge_condition="t.circuit_id = s.circuit_id",
    columns_to_update=[
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)


In [0]:
%sql
SELECT *
FROM delta.`s3://bizf1project123/Silver/Kalashri_increment/circuits`;

In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Silver/Kalashri_increment/circuits",
#     True
# )

In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Silver/Kalashri_increment/circuits",
#     True
# )